# AnomalyGPT Google Colab Setup Notebook

This notebook provides a step-by-step guide to set up and run AnomalyGPT on Google Colab.

For detailed explanations, see [COLAB_SETUP_GUIDE.md](./COLAB_SETUP_GUIDE.md)

**Estimated Setup Time**: 1-2 hours (depending on download speeds)

**GPU Requirement**: T4 (minimum), A100 (recommended)

## Step 1: Check GPU and Environment

In [ ]:
# Check GPU availability
!nvidia-smi

import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## Step 2: Mount Google Drive (Optional but Recommended)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create workspace directory
import os
WORKSPACE = '/content/drive/MyDrive/AnomalyGPT_workspace'
os.makedirs(WORKSPACE, exist_ok=True)
print(f"Workspace created at: {WORKSPACE}")

## Step 3: Clone Repository

In [ ]:
# Clone repository
!git clone https://github.com/CASIA-IVA-Lab/AnomalyGPT.git
%cd AnomalyGPT

## Step 4: Install Dependencies

In [ ]:
# Install PyTorch with CUDA 11.7
!pip install -q torch==1.13.1+cu117 torchvision==0.14.1+cu117 torchaudio==0.13.1+cu117 \
    --extra-index-url https://download.pytorch.org/whl/cu117

# Install other requirements
!pip install -q deepspeed==0.9.2 transformers==4.29.1 peft==0.3.0 sentencepiece
!pip install -q einops==0.6.1 timm==0.6.7 opencv-python==4.8.0.74 scikit-learn==1.3.0
!pip install -q gradio==3.41.2 kornia==0.7.0 ftfy==6.1.1 regex==2022.10.31
!pip install -q Pillow==10.0.0 easydict==1.10 iopath==0.1.10 pytorchvideo==0.1.5
!pip install -q matplotlib==3.7.2 tqdm==4.64.1

print("\n✓ Dependencies installed successfully!")

In [ ]:
# Verify installation
import torch
import transformers
import deepspeed
import peft

print(f"PyTorch: {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"DeepSpeed: {deepspeed.__version__}")
print(f"PEFT: {peft.__version__}")

## Step 5: Download Pre-trained Checkpoints

### 5.1 ImageBind Checkpoint (~5GB)

In [ ]:
!mkdir -p pretrained_ckpt/imagebind_ckpt
!wget -O pretrained_ckpt/imagebind_ckpt/imagebind_huge.pth \
    https://dl.fbaipublicfiles.com/imagebind/imagebind_huge.pth

# Verify
!ls -lh pretrained_ckpt/imagebind_ckpt/

### 5.2 PandaGPT Checkpoint (~13GB)

In [ ]:
!mkdir -p pretrained_ckpt/pandagpt_ckpt/7b
!wget -O pretrained_ckpt/pandagpt_ckpt/7b/pytorch_model.pt \
    https://huggingface.co/openllmplayground/pandagpt_7b_max_len_1024/resolve/main/pytorch_model.pt

# Verify
!ls -lh pretrained_ckpt/pandagpt_ckpt/7b/

### 5.3 Vicuna Checkpoint

**Important**: Vicuna requires LLaMA base weights. Follow instructions in [COLAB_SETUP_GUIDE.md Section 3.2](./COLAB_SETUP_GUIDE.md#32-vicuna-checkpoint).

For this demo, we'll assume you have:
1. Obtained LLaMA weights from Meta
2. Converted to HuggingFace format
3. Combined with Vicuna delta weights

If you already have the combined Vicuna weights, upload or link them:

In [ ]:
# Option 1: If you have Vicuna weights in Google Drive
# !ln -s /content/drive/MyDrive/vicuna_7b_v0 pretrained_ckpt/vicuna_ckpt/7b_v0

# Option 2: Download Vicuna delta and combine (requires LLaMA base)
# See COLAB_SETUP_GUIDE.md Section 3.2 for detailed instructions

# Verify Vicuna checkpoint structure
!ls pretrained_ckpt/vicuna_ckpt/7b_v0/ 2>/dev/null || echo "Vicuna checkpoint not found. Please follow setup guide."

## Step 6: Download MVTec-AD Dataset

**Option A: Manual Download**

1. Register and download from: https://www.mvtec.com/company/research/datasets/mvtec-ad
2. Upload `mvtec_anomaly_detection.tar.xz` to Colab
3. Run the extraction cell below

**Option B: If you have it in Google Drive**

In [ ]:
# If dataset is in Google Drive
# !ln -s /content/drive/MyDrive/mvtec_anomaly_detection data/mvtec_anomaly_detection

# Or extract from tar file
# !mkdir -p data
# !tar -xf mvtec_anomaly_detection.tar.xz -C data/

# Verify dataset
!ls data/mvtec_anomaly_detection/ 2>/dev/null || echo "MVTec-AD dataset not found. Please download first."

In [ ]:
# Verify dataset structure
import os

def verify_mvtec():
    root_dir = 'data/mvtec_anomaly_detection'
    if not os.path.exists(root_dir):
        print("❌ Dataset not found!")
        return
    
    categories = [d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))]
    print(f"✓ Found {len(categories)} categories: {', '.join(sorted(categories))}")
    
    # Check first category
    if categories:
        cat = categories[0]
        train_path = os.path.join(root_dir, cat, 'train', 'good')
        if os.path.exists(train_path):
            num_train = len(os.listdir(train_path))
            print(f"✓ Example: {cat} has {num_train} training images")
        else:
            print(f"❌ Training path not found: {train_path}")

verify_mvtec()

## Step 7: Download PandaGPT Training Data (Optional)

Required for full training. For quick testing, you can skip this.

In [ ]:
# Download JSON metadata
!mkdir -p data/images
!wget -O data/pandagpt4_visual_instruction_data.json \
    https://huggingface.co/datasets/openllmplayground/pandagpt_visual_instruction_dataset/resolve/main/pandagpt4_visual_instruction_data.json

# Note: Images are ~50GB. For testing, you can skip this and modify training script
# See COLAB_SETUP_GUIDE.md Section 7.6 for how to train without PandaGPT data

print("✓ PandaGPT metadata downloaded")
print("⚠ Image download skipped (50GB). See guide for full setup.")

## Step 8: Download Pre-trained AnomalyGPT Weights (Skip Training)

In [ ]:
# Download pre-trained weights to skip training
!mkdir -p code/ckpt/train_mvtec
!wget -O code/ckpt/train_mvtec/pytorch_model.pt \
    https://huggingface.co/FantasticGNU/AnomalyGPT/resolve/main/train_mvtec/pytorch_model.pt

# Verify
!ls -lh code/ckpt/train_mvtec/

## Step 9: Run Inference and Evaluation

In [ ]:
%cd /content/AnomalyGPT/code

# Run inference with 1-shot learning
!python test_mvtec.py --few_shot True --k_shot 1 --round 3

## Step 10: Launch Web Demo

In [ ]:
# First, modify web_demo.py to enable public sharing
!sed -i 's/demo.launch(share=False)/demo.launch(share=True)/' web_demo.py

# Launch demo
!python web_demo.py

## Optional: Training from Scratch

### Create Colab-friendly training script

In [ ]:
%%writefile code/train_mvtec_colab.sh
#!/bin/bash

deepspeed --include localhost:0 --master_port 28400 train_mvtec.py \
    --model openllama_peft \
    --stage 1 \
    --imagebind_ckpt_path ../pretrained_ckpt/imagebind_ckpt/imagebind_huge.pth \
    --vicuna_ckpt_path ../pretrained_ckpt/vicuna_ckpt/7b_v0/ \
    --delta_ckpt_path ../pretrained_ckpt/pandagpt_ckpt/7b/pytorch_model.pt \
    --max_tgt_len 1024 \
    --data_path ../data/pandagpt4_visual_instruction_data.json \
    --image_root_path ../data/images/ \
    --save_path ./ckpt/train_mvtec/ \
    --log_path ./ckpt/train_mvtec/log/

### Reduce epochs for testing

In [ ]:
# Modify config to reduce training time
!sed -i 's/epochs: 50/epochs: 5/' code/config/openllama_peft.yaml

# Verify
!grep -A 5 "train:" code/config/openllama_peft.yaml

### Start training

In [ ]:
%cd /content/AnomalyGPT/code

# Make script executable
!chmod +x train_mvtec_colab.sh

# Start training (this will take many hours)
!bash train_mvtec_colab.sh

## Troubleshooting

In [ ]:
# Check GPU memory usage
!nvidia-smi

In [ ]:
# View training logs
!tail -n 50 /content/AnomalyGPT/code/ckpt/train_mvtec/log/*.log 2>/dev/null || echo "No logs found"

In [ ]:
# Check checkpoint structure
!tree -L 3 pretrained_ckpt/

## Summary

After completing this notebook, you should have:

✓ Installed all dependencies  
✓ Downloaded all required checkpoints  
✓ Prepared MVTec-AD dataset  
✓ Run inference with pre-trained weights  
✓ (Optional) Trained your own model  
✓ (Optional) Launched web demo  

For detailed documentation, see [COLAB_SETUP_GUIDE.md](./COLAB_SETUP_GUIDE.md)